<a href="https://colab.research.google.com/github/ankitta-singh/machinelearning/blob/main/06_adaboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving house-prices-advanced-regression-techniques.zip to house-prices-advanced-regression-techniques.zip


In [2]:
import zipfile
with zipfile.ZipFile('house-prices-advanced-regression-techniques.zip', 'r') as zip_ref:
    zip_ref.extractall('house data')

In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
df = pd.read_csv('house data/train.csv')

In [12]:
x=df.drop('SalePrice',axis=1)
y=df['SalePrice']

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns

In [27]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [29]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

ada_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=4),
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    ))
])

In [30]:
ada_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea...
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object'))])),
                ('model',
                 AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=4),
                                   learning_rate=0.05, n_estimators=100,
                                   random_state=42))])

In [31]:
y_pred_ada = ada_model.predict(X_test)

In [32]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_ada = mean_absolute_error(y_test, y_pred_ada)
rmse_ada = np.sqrt(mean_squared_error(y_test, y_pred_ada))
r2_ada = r2_score(y_test, y_pred_ada)

print("AdaBoost Regression")
print("=" * 40)
print("MAE :", mae_ada)
print("RMSE:", rmse_ada)
print("R2  :", r2_ada)

AdaBoost Regression
MAE : 22631.907678527477
RMSE: 35207.231602320666
R2  : 0.8383968134317437


In [34]:
y_train_pred_ada = ada_model.predict(X_train)

print("TRAINING")
print("MAE :", mean_absolute_error(y_train, y_train_pred_ada))
print("RMSE:", np.sqrt(mean_squared_error(y_train, y_train_pred_ada)))
print("R2  :", r2_score(y_train, y_train_pred_ada))

print("\nTESTING")
print("MAE :", mean_absolute_error(y_test, y_pred_ada))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ada)))
print("R2  :", r2_score(y_test, y_pred_ada))

TRAINING
MAE : 19305.092912012202
RMSE: 25695.733306728867
R2  : 0.8893008411145417

TESTING
MAE : 22631.907678527477
RMSE: 35207.231602320666
R2  : 0.8383968134317437


In [42]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.1, 1.0],
    "model__estimator__max_depth": [2, 3, 4]
}

In [43]:
grid_search = GridSearchCV(
    estimator=ada_model,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

In [44]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


GridSearchCV(estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median'))]),
                                                                         Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'L...
       'SaleType', 'SaleCondition'],
      dtype='object'))])),
                                       ('model',
                                        AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=4),
                                                          learning_rate=0.05,
                                                          n_estimators=100,
                                                          random_state=42))]),
             n_jobs=-1,
             param_grid={'model__estimator__max_depth': [2, 3, 4],
                         'model__learning_rate': [0.01, 0.1, 1.0],
                         'model__n_estimators': [50, 100, 200]},
             scoring='neg_mean_squared_error', verbose=1)

In [46]:
print("Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'model__estimator__max_depth': 4, 'model__learning_rate': 1.0, 'model__n_estimators': 200}


In [47]:
print("Best CV RMSE:")
print(np.sqrt(-grid_search.best_score_))

Best CV RMSE:
33316.98674006798


In [48]:
best_ada = grid_search.best_estimator_
y_pred_best_ada = best_ada.predict(X_test)

In [49]:
mae_best = mean_absolute_error(y_test, y_pred_best_ada)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best_ada))
r2_best = r2_score(y_test, y_pred_best_ada)

print("Best AdaBoost Regression")
print("=" * 40)
print("MAE :", mae_best)
print("RMSE:", rmse_best)
print("R2  :", r2_best)

Best AdaBoost Regression
MAE : 22039.65928989809
RMSE: 31281.88712603461
R2  : 0.8724230970452002
